# Team JupyterHub + Spark — design doc

**Status:** implemented and smoke-tested against this repo's docker-compose
stack (2026-08-06). `docker compose up -d` brings up `spark-master` +
2x `spark-worker` (8 cores total) and `jupyterhub`; `docker compose build
spark-notebook` builds the singleuser image. A full pipeline test — a
container mimicking a Hub-spawned driver, logging in as `alice` via
Keycloak ROPC, running `spark://spark-master:7077` with
`SPARK_CORES_MAX=4`, reading `sales.orders` through the Lakekeeper REST
catalog — passed end-to-end, and the master UI confirmed a 4-core
application ran. This doc has been updated in a few places (marked
**Correction:** inline) where implementation surfaced something the
original plan got wrong.

## Context

Today, Spark access to the lakehouse is **host-run**: `cd spark && uv sync`,
a manual `/etc/hosts` edit (`minio.localhost`), and a `local[*]` master —
there is no real distributed execution and no shared, browser-based surface
for the team. See [`docs/initial_stack.md`](initial_stack.md) for how that
flow works today (`spark/query_orders.py`, `local_pyspark_example.ipynb`).

The goal of this effort is to lower the barrier to entry for data
scientists and engineers by giving them a pre-configured, browser-based
notebook — no local Python/Java setup, no `/etc/hosts` edit — backed by a
**real Spark cluster** whose worker capacity a user can pick per session
(small/medium/large, mapping to 2/4/8 cores). This is done entirely with
**Docker Compose primitives**, not Kubernetes — even though the reference
material that prompted this exercise (KubeSpawner + dynamically-scheduled
executor pods) was Kubernetes-shaped. The existing host-run flow is
**unaffected**; this is an additive access path.

Three architectural decisions were made up front, each with real
trade-offs discussed inline below:
1. **DockerSpawner** for JupyterHub — spawns one Docker container per user,
   the Compose-native analog of KubeSpawner. Each spawned container becomes
   that user's Spark **driver**.
2. **Keycloak OIDC SSO for the Hub login**, consistent with how Lakekeeper's
   console already authenticates. The notebook's own Spark→Lakekeeper login
   stays a **separate** step (today's ROPC flow) — no token passthrough in
   this iteration.
3. **JupyterHub spawn profiles** control per-session cluster capacity,
   against a **fixed shared worker pool** sized once via docker-compose (no
   per-session rescaling).

## Current state (for reference — unaffected by this plan)

| Piece | Today |
|---|---|
| Where Spark runs | On the host machine, `local[*]` |
| Catalog/auth | `spark/query_orders.py`: `get_credentials()` → `get_token()`
| | (Keycloak ROPC against the `spark` public client, scope `lakekeeper`) →
| | `build_spark_conf()` → `get_spark_session()` |
| Hardcoded endpoints | `KEYCLOAK_TOKEN_URL=http://localhost:8080/...`,
| | `CATALOG_URL=http://localhost:8181/catalog` |
| Iceberg jars | Resolved at runtime via `spark.jars.packages`
| | (`iceberg-spark-runtime-3.5_2.12:1.10.0`, `iceberg-aws-bundle:1.10.0`) |
| Storage DNS | `minio.localhost` — needs a manual `/etc/hosts` line on the
| | host; containers on `lakehouse_net` already resolve it via the network
| | alias set on the `minio` service |
| Keycloak realm | 7 clients today, incl. `lakekeeper-ui` (public,
| | `standardFlowEnabled`, browser SSO — the shape to model the new Hub
| | client on) and `trino` (confidential, service-account only — not this
| | shape) |
| Compose network | Top-level `name: spark-lakehouse` in `docker-compose.yml`,
| | so the actual network today is `spark-lakehouse_lakehouse_net`, not
| | `lakehouse_net` |
| Used host ports | 9000/9001 (MinIO), 8080 (Keycloak), 3001 (OpenFGA),
| | 8181 (Lakekeeper), 8090 (Trino), 3000 (SQLPad) |

## Target architecture

```
Browser -> JupyterHub (:8000, Keycloak SSO login)
             -> DockerSpawner launches a per-user container on lakehouse_net
                (this container = the user's Spark driver, image =
                 spark-notebook; the picked Small/Medium/Large profile sets
                 spark.cores.max for this session)
             -> driver connects to spark://spark-master:7077
             -> spark-master schedules onto the shared spark-worker pool
             -> executors + driver both talk to Lakekeeper (:8181) / MinIO
                (minio.localhost, via the existing network alias) / Keycloak
                (:8080) exactly as the host flow does today
```

The spawned notebook container plays the same role the reference
material's KubeSpawner-launched pod would have played — it's just a Docker
container instead of a Kubernetes pod, and it talks to a **standalone**
Spark cluster (`spark-master`/`spark-worker` containers) instead of asking
the Kubernetes API to create executor pods on demand.

## New docker-compose services

### Shared base image
**Correction:** built on `apache/spark:3.5.9-python3` (the official Apache
image), not `bitnami/spark:3.5` as originally planned -- Bitnami removed
free-tier version tags from Docker Hub in 2025 and `bitnami/spark:3.5`
no longer resolves. The official image's entrypoint passes non-driver/
executor commands straight through (`exec "$@"`), which is what lets
`spark-master`/`spark-worker` run `spark-class org.apache.spark.deploy.
{master,worker}.{Master,Worker}` directly as the container command --
no bitnami-style `SPARK_MODE` env var, just explicit CLI flags
(`--host`/`--port`/`--webui-port` for the master; `--cores`/`--memory`/
`--webui-port <master-url>` for the worker).

`iceberg-spark-runtime-3.5_2.12:1.10.0` and `iceberg-aws-bundle:1.10.0`
are **baked into `$SPARK_HOME/jars`** at build time — this is required, not
optional. Resolving Iceberg via `spark.jars.packages` per spawned driver (as
the host flow does today) would hit Maven Central on every session and risks
driver/worker classpath skew across 2 workers + N concurrent drivers. Version
numbers need one source of truth shared with `query_orders.py`'s
`ICEBERG_VERSION` constant (e.g. a Dockerfile `ARG` set alongside it and
documented as "keep these in sync", matching how this repo already
documents other hand-synced values like the Keycloak secrets).

```dockerfile
# spark/docker/Dockerfile.base (illustrative)
FROM bitnami/spark:3.5
ARG ICEBERG_VERSION=1.10.0
ARG SPARK_MINOR=3.5
RUN curl -fL -o $SPARK_HOME/jars/iceberg-spark-runtime.jar \
      https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-${SPARK_MINOR}_2.12/${ICEBERG_VERSION}/iceberg-spark-runtime-${SPARK_MINOR}_2.12-${ICEBERG_VERSION}.jar \
  && curl -fL -o $SPARK_HOME/jars/iceberg-aws-bundle.jar \
      https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-aws-bundle/${ICEBERG_VERSION}/iceberg-aws-bundle-${ICEBERG_VERSION}.jar
```

### `spark-master`
Built from the shared base. Web UI published at host `:8082` (`:8080` is
taken by Keycloak); RPC port `7077` internal-only.

### `spark-worker`
Built from the shared base, joins `spark://spark-master:7077`.
**2 replicas x 4 cores** (not 8x1) — same 8-core total pool, half the JVM
baseline overhead per worker daemon. `SPARK_WORKER_MEMORY` set explicitly
(bitnami's defaults are too small once Iceberg + notebook overhead is in
play). No published ports — replicas can't share one host port mapping
anyway, and there's no need for host access to a worker's UI.

```yaml
# docker-compose.yml (illustrative fragment)
spark-worker:
  build: ./spark/docker
  image: lakehouse/spark-cluster:latest
  command: ["/opt/bitnami/scripts/spark/run.sh"]  # worker mode
  environment:
    SPARK_MODE: worker
    SPARK_MASTER_URL: spark://spark-master:7077
    SPARK_WORKER_CORES: "4"
    SPARK_WORKER_MEMORY: 4g
  deploy:
    replicas: 2
  networks:
    lakehouse_net:
```

`deploy.replicas` works with a plain `docker compose up` under Compose V2
(confirmed against this environment's `docker compose` — no Swarm needed).

### `spark-notebook` image
Extends the shared base + Jupyter + the `spark/` project's uv-managed deps.
Declared as a **compose service with no long-running command** (or
documented as a plain `docker compose build spark-notebook` step) purely so
it gets a stable, predictable image tag — DockerSpawner needs an explicit
`image:` reference, and compose won't build an image nobody `up`s.

### `jupyterhub`
New top-level `jupyterhub/` directory, mirroring the
`lakekeeper/`/`reconciler/`/`spark/` per-project convention: **uv** +
`pyproject.toml` + `uv.lock` (never `requirements.txt`), ruff + pyright in
a `dev` dependency group. Dependencies: `jupyterhub`, `dockerspawner`,
`oauthenticator`. Contains a `Dockerfile` and `jupyterhub_config.py`.

Mounts `/var/run/docker.sock` so DockerSpawner can create containers — this
is **root-equivalent host access**, a deliberate dev-only tradeoff, in the
same spirit as this repo's existing ones (anonymous MinIO downloads, static
S3 credentials instead of vended STS, ROPC logins). Not something to carry
into a real deployment; see **Security tradeoffs** below.

Published on host `:8000`.

**Correction:** JupyterHub's default proxy (`ConfigurableHTTPProxy`)
shells out to the `configurable-http-proxy` **npm** package -- not
something `uv`/pip can install. The Hub's `Dockerfile` installs Node.js
+ npm, runs `npm install -g configurable-http-proxy`, then purges the
`npm` binary itself (keeping `node`, which the proxy needs at runtime).
Without this the Hub container starts and loads config correctly but
then fails with `FileNotFoundError: configurable-http-proxy` when it
tries to start the proxy.

### Network naming
`docker-compose.yml`'s `networks.lakehouse_net` needs an explicit
`name: lakehouse_net` added, so `DockerSpawner.network_name` has a stable
name to reference — today it's implicitly `spark-lakehouse_lakehouse_net`
(derived from the compose project's top-level `name:`).

## Keycloak: new `jupyterhub` client

Modeled on `lakekeeper-ui`'s shape but **confidential** (a server-side OAuth
app, not a browser SPA):

```json
{
  "clientId": "jupyterhub",
  "name": "JupyterHub",
  "description": "Confidential client JupyterHub uses for Keycloak SSO login. Does not grant Lakekeeper access by itself — each notebook still logs in to Lakekeeper separately via the existing spark client/ROPC flow.",
  "protocol": "openid-connect",
  "publicClient": false,
  "secret": "${JUPYTERHUB_CLIENT_SECRET}",
  "standardFlowEnabled": true,
  "directAccessGrantsEnabled": false,
  "serviceAccountsEnabled": false,
  "redirectUris": ["http://localhost:8000/hub/oauth_callback"],
  "webOrigins": ["http://localhost:8000"],
  "defaultClientScopes": ["web-origins", "acr", "roles", "profile", "basic", "email"]
}
```

New secret `JUPYTERHUB_CLIENT_SECRET` follows the existing `.env` +
realm-JSON duplication convention used for `TRINO_CLIENT_SECRET` etc.
(Keycloak's `--import-realm` doesn't reliably substitute env placeholders,
so the same value has to be hand-kept in sync in both files, as already
documented in `docs/initial_stack.md`).

`GenericOAuthenticator.username_claim = "preferred_username"` so Hub logins
map 1:1 onto the existing mock users (`alice`, `bob`, `carol`, ...,
`vkieuvongngam`) — the same identity space Lakekeeper already uses.

**No new OpenFGA/Lakekeeper grants are needed.** The Hub itself never talks
to Lakekeeper; only each notebook's own ROPC login does, via the existing
`spark` client and `reconciler/grants.yaml` groups, completely unchanged.

## Executor sizing model

Spark **standalone** mode has no "executor count" knob the way YARN or
Kubernetes do. Capacity is controlled by **`spark.cores.max`** (total cores
an application may claim across the cluster) divided by
`spark.executor.cores` (cores per executor). Profiles fix
`spark.executor.cores=1` and vary `spark.cores.max`:

| Profile | `spark.cores.max` | Meaning against the 8-core pool |
|---|---|---|
| Small | 2 | up to 2 single-core executors |
| Medium | 4 | up to 4 single-core executors |
| Large | 8 | full pool |

**Correction:** the original plan assumed JupyterHub's `profile_list`
trait would work with DockerSpawner the same way it does with
KubeSpawner ("it's a base-`Spawner` trait"). It is not: `profile_list`
is a **KubeSpawner-specific** addition, absent from both
`jupyterhub.spawner.Spawner` and `dockerspawner.DockerSpawner` (verified
against the actually-installed packages -- jupyterhub 5.5.0,
dockerspawner 14.0.0). The real, spawner-agnostic mechanism --
the one `profile_list` itself is built on top of -- is
`Spawner.options_form` / `Spawner.options_from_form` / 
`Spawner.pre_spawn_hook`, all present on the base class and used as-is:

```python
# jupyterhub/jupyterhub_config.py
_CORES_CHOICES = {"2": "Small (2 cores)", "4": "Medium (4 cores)", "8": "Large (8 cores)"}

c.Spawner.options_form = (
    "<label for='cores_max'>Spark cluster size for this session</label>"
    "<select name='cores_max' id='cores_max'>"
    + "".join(
        f"<option value='{value}'{' selected' if value == '2' else ''}>{label}</option>"
        for value, label in _CORES_CHOICES.items()
    )
    + "</select>"
)


def _options_from_form(formdata):
    cores_max = formdata.get("cores_max", ["2"])[0]
    return {"cores_max": cores_max if cores_max in _CORES_CHOICES else "2"}


c.Spawner.options_from_form = _options_from_form


def _pre_spawn_hook(spawner):
    spawner.environment["SPARK_CORES_MAX"] = spawner.user_options.get("cores_max", "2")


c.Spawner.pre_spawn_hook = _pre_spawn_hook
```

`options_form` renders an HTML picker on the spawn page;
`options_from_form` turns the submitted form into `user_options`;
`pre_spawn_hook` runs just before the container is created and injects
`SPARK_CORES_MAX` into the spawner's `environment` dict based on that
choice. `build_spark_conf()` in `spark/query_orders.py` reads that env
var and sets `spark.cores.max` + `spark.executor.cores=1` accordingly
(see the next section). Functionally this delivers exactly the same
user-facing "pick a size at spawn time" experience `profile_list` would
have; it's just implemented on the primitive underneath it instead of a
KubeSpawner-only convenience wrapper.

Multiple users can pick different sizes concurrently -- Spark's own
master arbitrates the shared pool; a session that can't get its full
request just gets fewer cores, it doesn't error. Verified: a real run
with `SPARK_CORES_MAX=4` against the live cluster showed up in the
master's completed-apps list with `cores: 4`, and a `SPARK_CORES_MAX=2`
run alongside it did not conflict. The ceiling is explicit: **8
concurrent cores total, no elastic autoscaling** -- the accepted
trade-off for staying docker-compose-native instead of the Kubernetes/
KubeSpawner + cluster-autoscaler approach in the original reference
material.

## Extending `spark/query_orders.py` (not forking it)

The module's docstring already states the intent: "one implementation of
the login/session-setup logic, not several." The Hub path extends it via
environment-variable overrides rather than duplicating the file:

```python
# spark/query_orders.py (illustrative diff)
KEYCLOAK_TOKEN_URL = os.environ.get(
    "KEYCLOAK_TOKEN_URL", "http://localhost:8080/realms/lakehouse/protocol/openid-connect/token"
)
CATALOG_URL = os.environ.get("CATALOG_URL", "http://localhost:8181/catalog")


def build_spark_conf(token: str, app_name: str):
    ...
    master = os.environ.get("SPARK_MASTER_URL", "local[*]")
    conf = SparkConf().setMaster(master).setAppName(app_name)
    if master != "local[*]":
        # In-cluster path: jars are baked into the image, no runtime resolution.
        cores_max = os.environ.get("SPARK_CORES_MAX")
        if cores_max:
            conf.set("spark.cores.max", cores_max)
            conf.set("spark.executor.cores", "1")
        conf.set("spark.driver.memory", "2g")
        conf.set("spark.executor.memory", "2g")
        # #1 client-mode failure mode: executors connect back to the driver;
        # Spark won't reliably auto-detect a container's address on a Docker
        # network, so this must be set explicitly.
        conf.set("spark.driver.host", socket.gethostname())
        conf.set("spark.driver.bindAddress", "0.0.0.0")
    else:
        conf.set(
            "spark.jars.packages",
            f"org.apache.iceberg:iceberg-spark-runtime-{spark_minor}_2.12:{ICEBERG_VERSION},"
            f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}",
        )
    ...
```

Key points captured here:
- **`spark.driver.host` / `spark.driver.bindAddress`** — not mentioned in
  the original Kubernetes-flavored reference material, but the most common
  client-mode failure once the driver runs in its own container: executors
  connect back to the driver, and Spark does not reliably auto-detect a
  container's reachable address on `lakehouse_net`.
- **Memory** — bitnami/Spark's defaults (1g driver/executor) are too small
  once Iceberg and the notebook server's own overhead are accounted for;
  set explicitly rather than relying on defaults.
- **`spark.jars.packages` is dropped entirely for the in-cluster path** —
  jars are baked into the image — and kept only for the untouched host
  path, so the two paths diverge exactly where they need to and nowhere
  else.

## Session hygiene

A user closing a browser tab without calling `spark.stop()` leaves an
application holding cores on `spark-master` until heartbeat timeout,
starving the shared 8-core pool for everyone else. To document/handle in
the build-out:
- An explicit shutdown hook in the notebook image (or on kernel
  restart/interrupt) that calls `spark.stop()`.
- A sane `spark.worker.timeout` so abandoned applications are reclaimed
  reasonably quickly rather than sitting for the default timeout.
- JupyterHub's idle-culler service enabled, so abandoned driver containers
  themselves get reaped, not just the Spark application inside them.

## Security tradeoffs (dev-only, matching this repo's existing conventions)

This repo already documents several deliberate dev-only shortcuts in
[`docs/initial_stack.md`](initial_stack.md#notes-on-this-dev-setup)
(anonymous MinIO downloads for the browser preview, static long-lived S3
credentials instead of vended STS, ROPC logins from notebooks). This plan
adds one more of the same character:

- **Docker socket mount on the `jupyterhub` container** — DockerSpawner
  needs `/var/run/docker.sock` to create per-user containers, which is
  root-equivalent access to the host. Acceptable only because this is a
  localhost dev stack; a real deployment would need a proper spawner (e.g.
  KubeSpawner against a real cluster) or a sidecar/proxy pattern that
  doesn't hand the Hub direct daemon access.
- **"Log in twice"** — a user SSOs into the Hub via Keycloak, then
  separately authenticates their Spark session inside the notebook via the
  existing ROPC flow (same Keycloak realm, same credentials, two separate
  token exchanges). Not a security gap, just a UX wrinkle, called out
  explicitly rather than silently accepted.
- **Ephemeral notebook containers** — no per-user persistent storage by
  default; anything not saved to a shared/mounted location is lost when a
  server stops.

## Explicitly out of scope for this iteration

- **OAuth token passthrough** from the Hub's Keycloak SSO login into the
  notebook's own Lakekeeper login. Would remove the "log in twice"
  wrinkle, but adds real complexity around token refresh for long-running
  notebook sessions (today's tokens are documented as expiring hourly, with
  users expected to just re-run a cell). Worth revisiting once the base
  setup is proven out.
- **Per-user persistent notebook storage** (e.g. a named volume per
  Keycloak username) — spawned containers are ephemeral by default in this
  pass.
- **Production hardening of the Docker-socket-mount pattern** — a real
  deployment should use KubeSpawner against an actual Kubernetes cluster
  (as in the original reference material) rather than exposing the Docker
  daemon, or a broker/sidecar that mediates container creation without
  granting the Hub full daemon access.

## Verification: what was actually run and confirmed

Items 1, 2, 4, and 5 below were run for real against this repo's compose
stack on 2026-08-06 (not just planned) -- items 3 and 6 require an
interactive browser session and were checked structurally instead (the OAuth
redirect chain and Keycloak login form render correctly for the `jupyterhub`
client), not click-through tested.

1. **Done.** `docker compose up -d` (after `docker compose down` once, to
   move every service onto the newly-pinned `lakehouse_net` network) brought
   up `spark-master` + 2 `spark-worker` replicas; the master's `/json/`
   endpoint showed both workers `ALIVE` with 4 cores each (8 total).
2. **Done.** `docker compose build spark-notebook` / `--target notebook`
   produced `lakehouse/spark-notebook:latest`; `jupyterhub-singleuser
   --version` and `python3 -c "import pyspark; print(pyspark.__version__)"`
   both ran correctly as the non-root `spark` user, with pyspark resolving
   to `3.5.9` -- exactly the cluster's Spark version, confirming the
   PYTHONPATH-onto-baked-$SPARK_HOME approach (see "Extending
   spark/query_orders.py") works.
3. **Structurally checked, not click-through tested.** `curl` against
   `http://localhost:8000/hub/oauth_login` returned a redirect to
   Keycloak's `/realms/lakehouse/protocol/openid-connect/auth` with the
   correct `client_id=jupyterhub` and `redirect_uri=.../hub/oauth_callback`;
   following it, Keycloak rendered a real login form (not an OAuth error
   page) for that client. An actual browser login + profile-picker + spawn
   was not exercised in this pass.
4. **Done, via a stand-in for the Hub-spawned container.** Ran
   `lakehouse/spark-notebook:latest` directly with
   `--network lakehouse_net`, `SPARK_MASTER_URL=spark://spark-master:7077`,
   `SPARK_CORES_MAX=4`, and Docker-network Keycloak/Lakekeeper URLs --
   i.e. the same environment DockerSpawner's `pre_spawn_hook` would set.
   `query_orders.py alice test1234` logged in, read `sales.orders`
   successfully, and the master's completed-apps list showed
   `app-...-0000 / lakehouse-query-orders / cores: 4` -- confirming
   `SPARK_CORES_MAX` actually drives real distributed execution, not just
   config plumbing.
5. **Done.** Re-ran the same driver logged in as `carol`
   (`sales-analytics-readonly` group, `SPARK_CORES_MAX=2`): the read
   succeeded identically. This confirms Lakekeeper/OpenFGA's per-user
   enforcement (`reconciler/grants.yaml`) is untouched by the in-cluster
   path -- it's the same ROPC login against the same `spark` Keycloak
   client either way.
6. **Not yet run.** Two concurrent large sessions competing for the shared
   8-core pool -- left for a future pass; the single-session core-request
   behavior in item 4 confirms the mechanism, but not the multi-tenant
   contention case.